# MK Praktikum Unggulan Universitas Gunadarma
# Nama Mata Kuliah: Praktikum Robotika Cerdas (Tingkat 4)  

---
# Pertemuan II

Pada praktikum ini, Anda akan melakukan generate gambar dan GIF menggunakan model GAN (Generative Adversarial Network)


## **1. Mengimport Library Python untuk GAN**

Pada tahapan ini Anda mendefinisikan terlebih dahulu library yang akan Anda gunakan.





In [ ]:
import warnings
warnings.filterwarnings("ignore")

In [ ]:
from keras.datasets import mnist

from tensorflow.keras import Sequential
from keras.layers import BatchNormalization
from keras.layers import Dense, Reshape, Flatten
from keras.layers import LeakyReLU
from tensorflow.keras.optimizers import Adam

import numpy as np
!mkdir generated_images

## **2. Mendefinisikan Variabel untuk Neural Network dan Data**


Pada tahap ini, didefinisikan beberapa variabel yang diperlukan dalam proses pembentukan dan pelatihan model GAN, di antaranya:
 - Ukuran gambar yang akan dihasilkan oleh algoritma GAN.
 - Channel warna yang digunakan pada gambar.
 - Variabel noise (latent) yang menjadi input awal bagi model untuk membentuk gambar.
 - Optimizer, yaitu variabel yang menentukan algoritma optimisasi yang digunakan selama proses pembelajaran. Pada tahap ini, digunakan algoritma Stochastic Gradient Descent (SGD) dengan learning rate sebesar 0.0001. Learning rate menentukan seberapa besar perubahan parameter model pada setiap proses pembelajaran. Nilai learning rate sebesar 0.0001 digunakan sebagai parameter dalam proses pelatihan model GAN.

In [ ]:
img_width = 28
img_height = 28
channels = 1
img_shape = (img_width, img_height, channels)
latent_dim = 100
adam = Adam(learning_rate=0.0001)

## **3. Membentuk Generator**


Generator adalah bagian dari GAN yang bertugas untuk belajar membuat data palsu dengan memasukkan umpan balik dari diskriminator. Generator belajar membuat diskriminator mengklasifikasikan outputnya sebagai yang sebenarnya atau nyata.





In [ ]:
def build_generator():
    model = Sequential()

    model.add(Dense(256, input_dim=latent_dim))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    model.add(Dense(256))
    model.add(LeakyReLU(alpha=0.2))
    model.add(BatchNormalization(momentum=0.8))

    # Membuat model menjadi ukuran 28x28x1
    model.add(Dense(int(np.prod(img_shape)), activation='tanh'))
    model.add(Reshape(img_shape))

    model.summary()
    return model

In [ ]:
generator = build_generator()

## **4. Membangun Discriminator**

Diskriminator dalam GAN hanyalah sebuah fungsi klasifikasi. Ia mencoba untuk membedakan data nyata dari data yang dibuat oleh generator. Diskriminator bisa menggunakan arsitektur jaringan apa pun yang sesuai dengan jenis data yang diklasifikasikan.

In [ ]:
def build_discriminator():
  model = Sequential()

  model.add(Flatten(input_shape=img_shape))
  model.add(Dense(512))
  model.add(LeakyReLU(alpha=0.2))
  model.add(Dense(256))
  model.add(Dense(1, activation='sigmoid'))

  model.summary()
  return model

discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy', optimizer=adam, metrics=['accuracy'])

## **5. Menghubungkan Discriminator dan Generator untuk membentuk GAN**



Pada tahapan ini Anda akan menghubungkan discriminator dan generator yang telah dibuat untuk membentuk sebuah GAN.

In [ ]:
GAN = Sequential()
discriminator.trainable = False
GAN.add(generator)
GAN.add(discriminator)

GAN.compile(loss='binary_crossentropy', optimizer=adam)
GAN.summary()

## **6. Menampilkan Output Gambar**

Berikut merupakan langkah untuk menyimpan output gambar. Hasil generate gambar akan tersedia pada folder <code>generated_images</code>.


In [ ]:
import matplotlib.pyplot as plt
import glob
import imageio
import PIL

save_name = 0.00000000

def save_imgs(epoch):
    r, c = 5, 5
    noise = np.random.normal(0, 1, (r * c, latent_dim))
    gen_imgs = generator.predict(noise)
    global save_name
    save_name += 0.00000001
    print("%.8f" % save_name)

    # Rescale images 0 - 1
    gen_imgs = 0.5 * gen_imgs + 0.5

    fig, axs = plt.subplots(r, c)
    cnt = 0
    for i in range(r):
        for j in range(c):
            axs[i,j].imshow(gen_imgs[cnt, :,:,0], cmap='gray')
            # axs[i,j].imshow(gen_imgs[cnt])
            axs[i,j].axis('off')
            cnt += 1
    fig.savefig("generated_images/%.8f.png" % save_name)
    print('saved')
    plt.close()

In [ ]:
save_imgs(1)

## **7. Training GAN**

Model akan melakukan pembelajaran sampai iterasi yang ditentukan. Anda dapat mengubah dan menyesuaikan nilai parameter <code>train</code>.

In [ ]:
def train(epochs, batch_size=64, save_interval=200):
  (X_train, _), (_, _) = mnist.load_data()

  # Rescale data between -1 and 1
  X_train = X_train / 127.5 -1.
  # X_train = np.expand_dims(X_train, axis=3)
  # print(X_train.shape)

  # Create our Y for our Neural Networks
  valid = np.ones((batch_size, 1))
  fakes = np.zeros((batch_size, 1))

  for epoch in range(epochs):
    # Get Random Batch
    idx = np.random.randint(0, X_train.shape[0], batch_size)
    imgs = X_train[idx]

    # Generate Fake Images
    noise = np.random.normal(0, 1, (batch_size, latent_dim))
    gen_imgs = generator.predict(noise)

    # Train discriminator
    d_loss_real = discriminator.train_on_batch(imgs, valid)
    d_loss_fake = discriminator.train_on_batch(gen_imgs, fakes)
    d_loss = 0.5 * np.add(d_loss_real, d_loss_fake)

    noise = np.random.normal(0, 1, (batch_size, latent_dim))

    # Inverse y label
    g_loss = GAN.train_on_batch(noise, valid)

    print("******* %d [D loss: %f, acc: %.2f%%] [G loss: %f]" % (epoch, d_loss[0], 100* d_loss[1], g_loss))

    if(epoch % save_interval) == 0:
      save_imgs(epoch)

  # print(valid)

train(1500, batch_size=64, save_interval=200)

## **8. Melakukan Generate GIF**

Setelah berhasil membuat gambar, model akan melakukan generate GIF.

In [ ]:
# Display a single image using the epoch number
# def display_image(epoch_no):
# return PIL.Image.open('generated_images/%.8f.png'.format(epoch_no))

anim_file = 'dcgan.gif'

with imageio.get_writer(anim_file, mode='I') as writer:
  filenames = glob.glob('generated_images/*.png')
  filenames = sorted(filenames)
  for filename in filenames:
    image = imageio.imread(filename)
    writer.append_data(image)
  image = imageio.imread(filename)
  writer.append_data(image)

# Instruksi Tugas Praktikum

1. Beri penjelasan dan logika program dari sintaks program yang terdapat pada poin-poin berikut:
   * Poin 6: Menampilkan Output Gambar,
   * Poin 7: Training GAN,
   * Poin 8: Making GIF
2. Silakan modifikasi kode pembelajaran menjadi 3000 parameter iterasi.
3. Apakah terdapat perbedaan dari hasil pembelajaran menggunakan parameter 1500 dan 3000? Jelaskan perbedaannya!


<br>

---

<br>

**CATATAN: DILARANG MENGGUNAKAN AI.** Silakan menjawab pertanyaan dengan analisis sendiri.

In [ ]:
# 1. Penjelasan dan Logika Program

"""
**Poin 6: Menampilkan Output Gambar**
Logika program pada Poin 6 bertujuan untuk menghasilkan dan menyimpan gambar menggunakan generator yang telah dilatih.
- Fungsi `save_imgs(epoch)` menghasilkan sekumpulan gambar baru dari distribusi noise acak (`np.random.normal(0, 1, (r * c, latent_dim))`).
- Generator memprediksi (`generator.predict`) citra berdasarkan input noise tersebut.
- Gambar-gambar yang dihasilkan (gen_imgs) akan di-rescale nilainya menjadi 0-1 untuk bisa ditampilkan sebagai citra.
- Matplotlib digunakan untuk menampilkan `r * c` (5x5 = 25) gambar dalam subplot dan menyimpannya sebagai file `.png` dalam folder `generated_images/` dengan penamaan berdasarkan urutan epoch.

**Poin 7: Training GAN**
Logika program pada Poin 7 adalah melatih Discriminator dan Generator secara bergantian dalam beberapa epoch.
- Pada setiap iterasi, diambil satu batch gambar nyata dari dataset MNIST (`idx = np.random.randint(0, X_train.shape[0], batch_size)`).
- Noise acak dimasukkan ke Generator untuk membuat sekumpulan gambar palsu (`gen_imgs`).
- Discriminator dilatih dengan menggunakan gambar nyata (label 1, `valid`) dan gambar palsu (label 0, `fakes`), loss-nya dihitung dan dirata-ratakan (`d_loss`).
- Kemudian, Generator dilatih menggunakan model gabungan GAN (`GAN.train_on_batch(noise, valid)`). Di tahap ini, Generator berusaha menipu Discriminator agar menghasilkan output bahwa gambar palsu adalah gambar nyata (label `valid`).
- Secara berkala (sesuai `save_interval`), fungsi `save_imgs` dipanggil untuk memantau perkembangan gambar yang dihasilkan oleh Generator.

**Poin 8: Melakukan Generate GIF**
Logika program pada Poin 8 bertujuan untuk menggabungkan seluruh gambar PNG yang disimpan selama proses training menjadi sebuah file animasi (GIF).
- Modul `glob` digunakan untuk membaca seluruh nama file dengan format `.png` di dalam direktori `generated_images/`.
- File-file tersebut kemudian diurutkan (`sorted`) agar animasi yang dihasilkan runtut sesuai waktu training.
- Dengan `imageio`, setiap file dibaca (`imageio.imread(filename)`) dan ditambahkan sebagai satu frame ke dalam file 'dcgan.gif' yang baru (`writer.append_data(image)`).
"""


In [ ]:
# 2. Modifikasi kode pembelajaran menjadi 3000 parameter iterasi.
train(3000, batch_size=64, save_interval=200)


In [ ]:
# 3. Perbedaan dari hasil pembelajaran menggunakan parameter 1500 dan 3000.

"""
**Penjelasan:**
Terdapat perbedaan signifikan pada hasil pembelajaran menggunakan iterasi 1500 dan 3000.
Dengan menambahkan jumlah iterasi (epoch) menjadi 3000, Generator mendapat lebih banyak kesempatan untuk belajar memperbarui bobot-bobotnya dari umpan balik (loss) Discriminator. Akibatnya, gambar digit yang dihasilkan Generator pada epoch 3000 akan terlihat jauh lebih jelas, tajam, dan realistis dibandingkan dengan hasil pada epoch 1500. Garis-garis pada gambar angka palsu semakin menyerupai bentuk angka asli dalam dataset MNIST, karena Generator semakin mahir dalam mereplikasi distribusi data aslinya dan menipu Discriminator.
"""




---


Copyright © Pengelola MK Praktikum Unggulan (Praktikum DGX), Universitas Gunadarma


https://www.praktikum-hpc.gunadarma.ac.id/